# AgriSmartAI - Notebook 05: Rule-Based Smart Irrigation Evaluation Engine
### Multi-Factor Atmospheric & Soil Moisture Decision Logic Verification

**Explicit Note**: Smart Irrigation in AgriSmartAI is implemented as a **Rule-Based Decision Engine** integrating live micro-climate parameters and soil moisture thresholds.

## 1. Setup & Rule Engine Interface

In [1]:
import os
import sys

def get_project_root():
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        return os.path.abspath(os.path.join(cwd, '..'))
    return cwd

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

def get_path(rel_path):
    clean_rel = rel_path.replace('../', '').replace('./', '')
    p1 = os.path.join(PROJECT_ROOT, clean_rel)
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(os.getcwd(), rel_path)
    if os.path.exists(p2):
        return p2
    return p1

import os
import sys
import pandas as pd
import numpy as np

sys.path.append('..')
from utils.models.irrigation_recommendation import SmartIrrigationEngine

engine = SmartIrrigationEngine()


## 2. Representative Micro-Climate Test Scenarios

In [2]:
scenarios = [
    {'name': 'Normal Baseline Conditions', 'moisture': 45.0, 'temp': 25.0, 'humidity': 60.0, 'rainfall': 0.0, 'expected': 'Moderate'},
    {'name': 'Heavy Rainfall Event', 'moisture': 45.0, 'temp': 24.0, 'humidity': 85.0, 'rainfall': 12.0, 'expected': 'Low'},
    {'name': 'High Soil Saturation', 'moisture': 85.0, 'temp': 26.0, 'humidity': 65.0, 'rainfall': 0.0, 'expected': 'Low'},
    {'name': 'Drought & Low Soil Moisture', 'moisture': 20.0, 'temp': 32.0, 'humidity': 40.0, 'rainfall': 0.0, 'expected': 'High'},
    {'name': 'Heatwave & High Evapotranspiration', 'moisture': 40.0, 'temp': 38.0, 'humidity': 35.0, 'rainfall': 0.0, 'expected': 'High'},
    {'name': 'High Atmospheric Humidity Suppressing Loss', 'moisture': 45.0, 'temp': 24.0, 'humidity': 90.0, 'rainfall': 0.0, 'expected': 'Low'}
]

results = []
for sc in scenarios:
    level, depth, msg = engine.evaluate_irrigation(
        moisture=sc['moisture'], temp=sc['temp'], humidity=sc['humidity'], rainfall=sc['rainfall']
    )
    status = 'Passed ✅' if level == sc['expected'] else 'Failed ❌'
    results.append({
        'Scenario': sc['name'],
        'Moisture (%)': sc['moisture'],
        'Temp (°C)': sc['temp'],
        'Humidity (%)': sc['humidity'],
        'Rainfall (mm)': sc['rainfall'],
        'Triggered Level': level,
        'Water Depth (mm)': depth,
        'Expected Level': sc['expected'],
        'Verification': status
    })

df_sc = pd.DataFrame(results)
display(df_sc)

,Scenario,Moisture (%),Temp (°C),Humidity (%),Rainfall (mm),Triggered Level,Water Depth (mm),Expected Level,Verification
0,Normal Baseline Conditions,45.0,25.0,60.0,0.0,Moderate,8-12,Moderate,Passed ✅
1,Heavy Rainfall Event,45.0,24.0,85.0,12.0,Low,0-5,Low,Passed ✅
2,High Soil Saturation,85.0,26.0,65.0,0.0,Low,0-5,Low,Passed ✅
3,Drought & Low Soil Moisture,20.0,32.0,40.0,0.0,High,15-20,High,Passed ✅
4,Heatwave & High Evapotranspiration,40.0,38.0,35.0,0.0,High,15-20,High,Passed ✅
5,High Atmospheric Humidity Suppressing Loss,45.0,24.0,90.0,0.0,Low,0-5,Low,Passed ✅


## 3. Live OpenWeatherMap API Integration Test

In [3]:
weather_data = engine.fetch_weather_data(city='Chennai')
print('Live Weather API Payload:')
print(weather_data)

Live Weather API Payload:
{'city': 'Chennai', 'temperature': 28.5, 'humidity': 55.0, 'rainfall': 0.0, 'condition': 'Partly Cloudy', 'wind_speed': 3.2, 'source': 'OpenWeatherMap Live API'}
